# NHOS WS Demo

This notebook demonstrates how to connect to the New Horizons Desktop Backend WebUI stream endpoint, subscribe to a specific `device_uid`, and continuously receive `snapshot` and `update` events.

The backend endpoint is `GET/WS /newhorizons/stream/ws`. Authentication works with `Authorization: Bearer <token>` or `?token=<token>`.

## Prerequisites

- The Desktop Backend is running, usually at `http://127.0.0.1:5051`
- You know a valid `username` and `password`
- You know the `device_uid` you want to subscribe to
- `requests` and `websocket-client` are installed

If an old `socketio` package happens to be installed, do not use it. This backend uses plain WebSocket, not Socket.IO.

In [ ]:
# If needed, install the dependencies first:
# %pip install requests websocket-client pandas

import json
from datetime import datetime

import requests
import websocket


In [ ]:
BASE_URL = "http://127.0.0.1:5051"
USERNAME = "admin"
PASSWORD = "admin"
DEVICE_UID = "3CDC7545CCD0"

WS_URL = BASE_URL.replace("http://", "ws://").replace("https://", "wss://") + "/newhorizons/stream/ws"
WS_URL

In [ ]:
resp = requests.post(
    f"{BASE_URL}/api/token",
    json={"username": USERNAME, "password": PASSWORD},
)
resp.raise_for_status()

token_payload = resp.json()
token = token_payload["token"]
expires_in = token_payload.get("expires_in")

print("token acquired successfully")
print(f"expires_in: {expires_in}")
print(f"token[:40]: {token[:40]}...")

## Open the stream connection

After the connection opens, the backend sends `hello_ack` first. Then we send `subscribe` for the target `device_uid`. If the subscription succeeds, you usually receive `subscribed`, and if the device already has cached data, you also receive a `snapshot`.

In [ ]:
ws = websocket.create_connection(
    WS_URL,
    header=[f"Authorization: Bearer {token}"],
    timeout=10,
)
ws.settimeout(1.0)

hello_ack = json.loads(ws.recv())
hello_ack

In [ ]:
ws.send(json.dumps({"type": "subscribe", "device_uid": DEVICE_UID}, separators=(",", ":")))

subscribed = json.loads(ws.recv())
subscribed

## Receive events continuously

The loop below stores each event in `events` and prints it immediately. The event format matches the backend implementation. Common event types include:

- `snapshot`: latest state after subscription
- `update`: incremental updates after the snapshot
- `stream_ended`: the device is no longer available or the subscription was removed
- `error`: message format or authorization error

In [ ]:
events = []

try:
    while True:
        raw = ws.recv()
        event = json.loads(raw)
        events.append({"received_at": datetime.now().isoformat(timespec="seconds"), **event})
        print(event)
except KeyboardInterrupt:
    print("Stream stopped manually")
except websocket.WebSocketTimeoutException:
    print("Timed out waiting for new events")
finally:
    ws.close()
    print("WebSocket closed")

## Convert to a table

If you want to do more analysis, convert `events` into a `pandas.DataFrame`. For `snapshot` and `update` events, the payload is usually under the `data` field.

In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(events)
    df.head()
except ImportError:
    print("pandas is not installed, skipping DataFrame conversion")

## Notes

If you need to connect to a remote environment, change `BASE_URL` to the actual address. If you are using HTTPS, the notebook automatically switches to `wss://`. If no data arrives, first confirm that `DEVICE_UID` is correct and that the backend has a usable visualization snapshot for that device.